# Area affected by droughts (time series)


In [ ]:
# general
from pathlib import Path
import pandas as pd
import numpy as np
import yaml 
import os

# plotting
import plotly.graph_objects as go
from   plotly.subplots import make_subplots
from PIL import Image
import io

wd = Path('/home/rooda/OneDrive/Projects/Whatif-Andes')
os.chdir(wd)

# import config and local module
with open("config.yml") as f:
    cfg = yaml.safe_load(f)

In [ ]:
# Regions and Scenarios
regions = {
    'Outer tropics': (-20, -8),
    'Desert Andes': (-30, -20),
    'Central Andes': (-35, -30),
    'North Patagonia': (-45, -35),
    'South Patagonia': (-52, -45),
    'Fuegian Andes': (-56, -52)}

scenarios = [
    '+1.5ºC (N=7)', 
    '+2.8ºC (N=5)', 
    '+4.0ºC (N=6)']

colors = ['#2E86AB', '#F18F01', "#C7231D"] 

## Data

In [ ]:
# data from drought analysis
df = pd.read_csv("data/hydro/droughts_hydroatlas_area_ts.csv", index_col=["scenario", "date"], parse_dates=["date"])
df = df.rolling(window=90, min_periods=1, center=True).mean()
df.index = df.index.set_levels(
    df.index.levels[0].map({
        'historical': 'Historical',
        '15C': scenarios[0],
        '28C': scenarios[1],
        '40C': scenarios[2]}),
    level=0)

# latitudinal values
df_lat = df.groupby(level='scenario').mean()
df_lat = df_lat.transpose()
df_lat.index = df_lat.index.astype(int)

# regional values 
df_agg = pd.DataFrame(index=df.index)
for region, (lat_min, lat_max) in regions.items():
    cols = [str(c) for c in df.columns if lat_min <= int(c) < lat_max]
    df_agg[region] = df[cols].sum(axis=1)

# total values (for animation)
df_all = df_agg.sum(axis=1)
df_all = df_all.reset_index().pivot_table(index='date', columns='scenario')
df_all.columns = df_all.columns.get_level_values(1)

for scenario in scenarios:
    df_all[scenario] = ((df_all[scenario] / df_all["Historical"]) -1)*100
    df_lat[scenario] = ((df_lat[scenario] / df_lat["Historical"]) -1)*100

## Plot (MS)

In [ ]:
fig = make_subplots(rows=3, cols=2, vertical_spacing=0.04, horizontal_spacing=0.08,
                    shared_xaxes=True,
                    column_widths=[0.35, 0.65],
                    specs=[[{"rowspan": 3}, {"secondary_y": True}], 
                           [None, {"secondary_y": True}], 
                           [None, {"secondary_y": True}]])

# Build categorical x-axis (daily as strings) and yearly tick labels
date_idx = df_agg[region].loc["Historical"].index
cat_array = date_idx.strftime('%Y-%m-%d').tolist()
year_idx = pd.date_range(date_idx.min().replace(month=1, day=1),
                         date_idx.max().replace(month=1, day=1),
                         freq='YS')
year_tickvals = year_idx.strftime('%Y-%m-%d').tolist()
year_labels = year_idx.strftime('%Y').tolist()

# Panel 1: Latitudinal bars (left column)
for scenario, color in zip(scenarios[::-1], colors[::-1]):
    fig.add_trace(go.Bar(x=df_lat[scenario], y=df_lat.index, 
                         marker_color=color, orientation='h',
                         name=scenario, showlegend=True), 
                  row=1, col=1)


# Panel 2-4: Regional time series (right column)
region_rows = [
    ('Outer tropics', 1),
    ('Central Andes', 2),
    ('South Patagonia', 3)
]

for region, row in region_rows:
    # What-if change
    what_if = (df_agg[region].loc[scenarios[1]] - df_agg[region].loc["Historical"]) / 1e3
    fig.add_trace(go.Scatter(x=what_if.index.strftime('%Y-%m-%d'), y=what_if.values,
                             mode='lines', fill='tozeroy',
                             name=scenarios[1] if row == 1 else None,
                             line=dict(color=colors[1], width=0.1),
                             showlegend=False),
                  row=row, col=2, secondary_y=True)
    
    # Historical
    fig.add_trace(go.Scatter(x=df_agg[region].loc["Historical"].index.strftime('%Y-%m-%d'),
                             y=df_agg[region].loc["Historical"].values / 1e3,
                             mode='lines', name='Historical' if row == 1 else None,
                             line=dict(color="#242424", width=1),
                             showlegend=(row == 1)),
                  row=row, col=2)

# Update axes
fig.update_yaxes(ticksuffix="ºS", dtick=5, range=(-56, -7), row=1, col=1)
fig.update_xaxes(title="Relative change (%)", title_standoff = 0, range=(0, 150), row=1, col=1)

fig.update_yaxes(range=[0, 80], row=1, col=2, secondary_y=False)
fig.update_yaxes(range=[0, 20], row=1, col=2, secondary_y=True)
fig.update_yaxes(title="Drought extent (10³ km²)", range=[0, 30], 
                 title_standoff=2, row=2, col=2, secondary_y=False)
fig.update_yaxes(title="Absolute change (10³ km²) <span style='color:lightgrey'>[current policies]</span>", 
                 range=[0, 6], title_standoff=10, row=2, col=2, secondary_y=True)
fig.update_yaxes(range=[0, 80], row=3, col=2, secondary_y=False)
fig.update_yaxes(range=[0, 40], row=3, col=2, secondary_y=True)

fig.update_xaxes(ticks="outside", griddash="dot", zeroline=False)
fig.update_yaxes(ticks="outside", griddash="dot", zeroline=False)

for row in [1, 2, 3]:
    fig.update_xaxes(
        type='category',
        categoryorder='array',
        categoryarray=cat_array,
        tickmode='array',
        tickvals=year_tickvals[::5],   # only every 5 years
        ticktext=year_labels[::5],     # show every 5th year only
        ticks="outside",
        griddash="dot", row=row, col=2)

# Add annotations
annotations = [
    ("a)", 0.27, 0.99),
    ("b) Outer tropics", 0.39, 0.99),
    ("c) Central Andes", 0.39, 0.65),
    ("d) South Patagonia", 0.39, 0.30)
]
for text, x, y in annotations:
    fig.add_annotation(text=text, x=x, y=y, xref="paper", yref="paper",
                       showarrow=False, font=dict(size=14, family="Arial", color="black"),
                       xanchor="left", yanchor="top")

# Add vertical line on the left
fig.add_shape(
    type="line",
    x0=-0.11, x1=-0.11,
    y0=-0.02, y1=1.0,
    xref="paper", yref="paper",
    line=dict(color="black", width=1)
)

# Add horizontal lines at region boundaries
for boundary_lat in [-7, -20, -30, -35, -45, -52, -57]:
    y_pos = (boundary_lat - (-56)) / ((-7) - (-56))
    fig.add_shape(
        type="line",
        x0=-0.12, x1=-0.10,
        y0=y_pos, y1=y_pos,
        xref="paper", yref="paper",
        line=dict(color="black", width=1)
    )

# Add region annotations on the vertical line
for region, (lat_min, lat_max) in regions.items():
    # Calculate midpoint of the region in latitude

    mid_lat = (lat_min + lat_max) / 2
    y_pos = (mid_lat - (-56)) / ((-7) - (-56))
    
    fig.add_annotation(
        text=region.replace(' ', '<br>'),
        bgcolor="white", 
        x=-0.11,
        y=y_pos,
        xref="paper",
        yref="paper",
        showarrow=False,
        font=dict(size=13, family="Arial"),
        xanchor="center",
        yanchor="middle",
    )

fig.update_layout(legend=dict(y=0.95, x=0.13, bgcolor="rgba(255,255,255,0.8)"),
                  autosize=False, width=900, height=600, barmode='overlay',
                  plot_bgcolor="rgba(229,236,246,0.8)",
                  margin=dict(l=130, r=10, b=10, t=10),
                  font=dict(size=13, family="Arial"))

fig.write_image("figures/fig_whatif_drought_extent.svg", height=600, width=900)
fig.show()

## Plot (animation)

In [ ]:
# Build categorical x-axis (daily as strings) and yearly tick labels
date_idx = df.index
cat_array = date_idx.strftime('%Y-%m-%d').tolist()
year_idx = pd.date_range(date_idx.min().replace(month=1, day=1),
                         date_idx.max().replace(month=1, day=1),
                         freq='YS')
year_tickvals = year_idx.strftime('%Y-%m-%d').tolist()
year_labels = year_idx.strftime('%Y').tolist()

# Store images for GIF
images = []

# Panel 2: Stacked area plot for scenarios
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.07, 
                    shared_xaxes=True, shared_yaxes=False)

# Panel 1: Historical drought extent
fig.add_trace(go.Scatter(
    x=df.index.strftime('%Y-%m-%d'),
    y=df["historical"] / 1000,
    mode='lines', 
    line=dict(width=0.5, color="#414141"),
    fill='tozeroy',
    fillcolor='rgba(40, 40, 40, 0.2)',
    showlegend=False,
), row=1, col=1)

# Panel 2: Stacked scenario changes
base = np.zeros_like(df[scenarios[0]])
for i, (scenario, color) in enumerate(zip(scenarios, colors)):
    # Calculate the incremental change for this scenario
    if i == 0:
        upper = df[scenario].values
    else:
        upper = df[scenario].values - df[scenarios[i-1]].values

    fig.add_trace(go.Scatter(
        x=pd.to_datetime(df.index.values).strftime('%Y-%m-%d'),
        y=base + upper,
        name=scenario,
        mode='lines',
        fill='tonexty',
        line=dict(color=color, width=0.5),
        fillcolor=color,
        showlegend=True,
    ), row=2, col=1)
    base = base + upper

    fig.update_xaxes(
        type='category',
        categoryorder='array',
        categoryarray=cat_array,
        tickmode='array',
        tickvals=year_tickvals,   # only Jan-01 each year
        ticktext=year_labels,     # show year only
        ticks="outside",
        griddash="dot",
    )

    fig.update_yaxes(title = "Drought extent (10³ km²)", range = [0, 250], row = 1, col = 1)
    fig.update_yaxes(title = "What-if change (%)", range = [0, 80], dtick= 20, ticksuffix = "%", row = 2, col = 1)
    fig.update_yaxes(ticks="outside", griddash="dot")

    # Layout with increased bottom margin for x-axis labels
    fig.update_layout(font_size = 14, font_family = "Arial", 
        height=600, width=1100,
        template="plotly",
        margin=dict(l=10, r=10, b=10, t=10),  # Increase bottom margin
        legend=dict(bgcolor='rgba(255,255,255,0.3)',
            yanchor="bottom", y=0.30,
            xanchor="right", x=0.99,
        )
    )
    
    # Convert figure to image and add to list
    img_bytes = fig.to_image(format="png", scale=4, height=600, width=1100)
    img = Image.open(io.BytesIO(img_bytes))
    images.append(img)

# Create GIF
images[0].save(
    'figures/fig_whatif_drought_extent.gif',
    save_all=True,
    append_images=images[1:],
    duration=2000,  # milliseconds per frame
    loop=0
)